In [ ]:
import numpy as np
import torch
import torch as tr
import torch.nn as nn
import math
import torch.nn.functional as F
from typing import Any, Tuple, List
from scipy.io import loadmat
import os
import pickle
from scipy.linalg import fractional_matrix_power
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random
from tqdm import tqdm
import logging
from math import log2
import datetime
import matplotlib.pyplot as plt
import copy

SAMPLING_RATE = 250
WINDOW_LENGTH = 250
SLIDING_STEP = 250
TEST_SLIDING_STEP = 250
DATA_PATH = '../../database/'

def Entropy(input_):
    epsilon = 1e-5
    entropy = -input_ * tr.log(input_ + epsilon)
    entropy = tr.sum(entropy, dim=1)
    return entropy

def band_filter(x):
    return x

def EA_alignment(X, return_ref=False):
    n_trials, n_channels, n_times = X.shape
    cov_matrices = np.array([np.cov(trial) for trial in X])
    mean_cov = np.mean(cov_matrices, axis=0)
    transform = fractional_matrix_power(mean_cov, -0.5)
    X_aligned = np.array([transform @ trial for trial in X])
    if return_ref:
        return X_aligned, mean_cov
    else:
        return X_aligned

def process_data(rawdata, mode, window_length, step, apply_EA=False, perform_fft=True, return_ref=False):
    channels = [60,61,62,54,55,56,53,47,57]
    slice_start = round(5/0.2)
    slice_end = round(55/0.2)+1
    raw_segments = []
    label_list = []
    data_length = 1250
    for block in range(6):
        for trial in range(40):
            filtered = [band_filter(rawdata[ch,160:1410,trial,block]) for ch in channels]
            for start_idx in range(0, data_length-window_length+1, step):
                segment = []
                for sig in filtered:
                    segment.append(sig[start_idx:start_idx+window_length])
                raw_segments.append(np.array(segment))
                label_list.append(trial)
    raw_segments = np.array(raw_segments)
    if apply_EA:
        if return_ref:
            aligned_segments, ref = EA_alignment(raw_segments, return_ref=True)
        else:
            aligned_segments = EA_alignment(raw_segments)
    else:
        aligned_segments = raw_segments
        if return_ref:
            ref = None
    if not perform_fft:
        if return_ref:
            return aligned_segments, np.array(label_list), ref
        else:
            return aligned_segments, np.array(label_list)
    else:
        data_list = []
        for segment in aligned_segments:
            trial_channels = []
            for sig in segment:
                fft_res = np.fft.rfft(sig, n=1250)/window_length
                if mode=="abs":
                    fft_res = np.abs(fft_res)[slice_start:slice_end]
                elif mode=="comp":
                    fft_res = np.concatenate((np.real(fft_res)[slice_start:slice_end],
                                              np.imag(fft_res)[slice_start:slice_end]))
                else:
                    raise ValueError("Invalid mode")
                trial_channels.append(fft_res)
            data_list.append(np.array(trial_channels))
        data_list = np.array(data_list)
        data_list = np.expand_dims(data_list, axis=-1)
        if return_ref:
            return data_list, np.array(label_list), ref
        else:
            return data_list, np.array(label_list)

def get_cache_filename(mode, subject, time_length):
    os.makedirs("cache", exist_ok=True)
    return os.path.join("cache", f"cache_{mode}{subject}{time_length}.pkl")

def data_batch_FFT_one_tester(mode, test_trainer, time_length, apply_EA=False, return_raw_test=False):
    cache_file = get_cache_filename(mode, test_trainer, time_length)
    if os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    subjects = [f"S{i}" for i in range(1,36)]
    window_length = int(SAMPLING_RATE * time_length)
    train_data_list, train_labels_list, train_refs = [], [], []
    test_index = subjects.index(test_trainer)
    train_subjects = subjects[:test_index] + subjects[test_index+1:]
    for subj in train_subjects:
        rawdata = loadmat(DATA_PATH + subj + '.mat')['data']
        d, l, ref = process_data(rawdata, mode, window_length, SLIDING_STEP, apply_EA=True, perform_fft=True, return_ref=True)
        train_data_list.append(d)
        train_labels_list.append(l)
        train_refs.append(ref)
    train_data = np.concatenate(train_data_list, axis=0)
    train_labels = np.concatenate(train_labels_list, axis=0)
    train_ref = np.mean(np.array(train_refs), axis=0)
    rawdata = loadmat(DATA_PATH + test_trainer + '.mat')['data']
    if return_raw_test:
        test_data, test_labels = process_data(rawdata, mode, window_length, TEST_SLIDING_STEP, apply_EA=False, perform_fft=False)
    else:
        test_data, test_labels = process_data(rawdata, mode, window_length, TEST_SLIDING_STEP, apply_EA=apply_EA, perform_fft=True)
    data_tuple = (train_data, train_labels, test_data, test_labels, train_ref)
    with open(cache_file, "wb") as f:
        pickle.dump(data_tuple, f)
    return data_tuple

def data_batch_FFT_one_tester_abs_UI(test_trainer, time_length):
    return data_batch_FFT_one_tester("abs", test_trainer, time_length, apply_EA=True, return_raw_test=True)

def EA_online(x, R, sample_num, weight=500):
    cov = np.cov(x)
    new_sample_count = sample_num + weight
    refEA = (R * sample_num + weight * cov) / new_sample_count
    return refEA

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class RobustBN(nn.Module):
    @staticmethod
    def find_bns(parent, alpha):
        replace_mods = []
        if parent is None:
            return []
        for name, child in parent.named_children():
            if isinstance(child, nn.BatchNorm2d):
                module = RobustBN(child, alpha)
                replace_mods.append((parent, name, module))
            else:
                replace_mods.extend(RobustBN.find_bns(child, alpha))
        return replace_mods
    @staticmethod
    def adapt_model(model, alpha):
        replace_mods = RobustBN.find_bns(model, alpha)
        for (parent, name, child) in replace_mods:
            setattr(parent, name, child)
        return model
    def __init__(self, bn_layer: nn.BatchNorm2d, momentum):
        super(RobustBN, self).__init__()
        self.num_features = bn_layer.num_features
        self.momentum = momentum
        if bn_layer.track_running_stats and bn_layer.running_var is not None and bn_layer.running_mean is not None:
            self.register_buffer("source_mean", bn_layer.running_mean.clone())
            self.register_buffer("source_var", bn_layer.running_var.clone())
        self.register_parameter("weight", nn.Parameter(bn_layer.weight.clone()))
        self.register_parameter("bias", nn.Parameter(bn_layer.bias.clone()))
        self.eps = bn_layer.eps
    def forward(self, x, trial_lengths=None):
        if self.training:
            b_var, b_mean = torch.var_mean(x, dim=[0,2,3], unbiased=False, keepdim=False)
            mean = (1-self.momentum)*self.source_mean + self.momentum*b_mean
            var = (1-self.momentum)*self.source_var + self.momentum*b_var
            self.source_mean = mean.detach().clone()
            self.source_var = var.detach().clone()
            mean, var = mean.view(1,-1,1,1), var.view(1,-1,1,1)
        else:
            mean, var = self.source_mean.view(1,-1,1,1), self.source_var.view(1,-1,1,1)
        x = (x-mean)/torch.sqrt(var+self.eps)
        weight = self.weight.view(1,-1,1,1)
        bias = self.bias.view(1,-1,1,1)
        return x*weight + bias

def update_bn_statistics(model, dataloader, bn_update_choice, device):
    model.eval()
    if bn_update_choice=="bn1":
        model.bn1.train()
        model.bn2.eval()
    elif bn_update_choice=="bn2":
        model.bn2.train()
        model.bn1.eval()
    else:
        model.bn1.train()
        model.bn2.train()
    with torch.no_grad():
        for batch_data, _ in dataloader:
            batch_data = batch_data.to(device)
            _ = model(batch_data)
    model.eval()

class CNNModel(nn.Module):
    def __init__(self, input_width, decay, num_classes=40):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(1,18,(9,1),bias=False)
        self.bn1 = nn.BatchNorm2d(18)
        self.dropout1 = nn.Dropout(0.25)
        self.conv2 = nn.Conv2d(18,18,(1,15),bias=False)
        self.bn2 = nn.BatchNorm2d(18)
        self.dropout2 = nn.Dropout(0.25)
        self.flat_features = 18*(input_width-14)
        self.fc = nn.Linear(self.flat_features, num_classes, bias=False)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            nn.init.normal_(m.weight,0.0,0.01)
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

def itr(n, p, t):
    if p<1.0/n:
        return 0.0
    elif p==1:
        return log2(n)*60.0/t
    else:
        return (log2(n)+p*log2(p)+(1.0-p)*log2((1.0-p)/(n-1)))*60.0/t

def main():
    set_seed(2025)
    MODE = "abs"
    if MODE=="abs":
        input_width = 251
        data_func = data_batch_FFT_one_tester_abs_UI
    else:
        input_width = 502
        data_func = lambda s,t: data_batch_FFT_one_tester("comp",s,t,apply_EA=True,return_raw_test=True)
    cache_dir = f"cache_{MODE}"
    model_dir = f"model_{MODE}"
    log_dir = f"CNN_result_{MODE}"
    os.makedirs(cache_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    logging.basicConfig(filename=os.path.join(log_dir, f"CNN_log_{MODE}_{timestamp}.txt"), level=logging.INFO, format='%(asctime)s %(message)s')
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    use_robustbn = True
    bn_update_choice = "both"
    robustbn_momentum = 0.7
    n_classes = 40
    window_length = 250
    slice_start = round(5/0.2)
    slice_end = round(55/0.2)+1
    subjects_to_plot = ['S1','S2','S3']
    curves = {}
    for subject in subjects_to_plot:
        train_datas, train_labels, _, _, train_ref = data_func(subject,1)
        train_datas = np.transpose(train_datas,(0,3,1,2))
        train_datas = torch.tensor(train_datas,dtype=torch.float32)
        train_labels = torch.tensor(train_labels,dtype=torch.long)
        train_dataset = TensorDataset(train_datas,train_labels)
        best_accuracy = 0.0
        best_model = None
        for run in tqdm(range(1), desc=f"{subject} Runs"):
            set_seed(2025+run)
            g = torch.Generator(); g.manual_seed(2025+run)
            train_loader = DataLoader(train_dataset,batch_size=512,shuffle=True,generator=g)
            model = CNNModel(input_width=input_width,decay=0.0001,num_classes=40).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.SGD(model.parameters(),lr=0.001,momentum=0.9,weight_decay=0.0001)
            model.train()
            for epoch in tqdm(range(50), desc=f"{subject} Run {run+1} Epochs"):
                for batch_data,batch_labels in train_loader:
                    batch_data,batch_labels = batch_data.to(device),batch_labels.to(device)
                    optimizer.zero_grad()
                    outputs = model(batch_data)
                    loss = criterion(outputs,batch_labels)
                    loss.backward()
                    optimizer.step()
            model.eval()
            R = train_ref.copy()
            _, _, test_datas, test_labels, _ = data_batch_FFT_one_tester("abs",subject,1,apply_EA=False,return_raw_test=True)
            y_true_list,y_pred_list = [],[]
            zk_arrs = torch.zeros(n_classes,dtype=torch.float32,device=device)
            pred_thresh = 0.7
            optimizer = optim.SGD(model.parameters(),lr=0.00003,momentum=0.9,weight_decay=0.0001)
            for i in range(len(test_datas)):
                sample = test_datas[i]
                R = EA_online(sample,R,i,5000)
                sqrtRefEA = fractional_matrix_power(R,-0.5)
                aligned_sample = np.dot(sqrtRefEA,sample)
                channel_features = []
                for j in range(aligned_sample.shape[0]):
                    sig = aligned_sample[j,:]
                    fft_j = np.fft.rfft(sig,n=1250)/window_length
                    feat_j = np.abs(fft_j)[slice_start:slice_end]
                    channel_features.append(feat_j)
                fft_res = np.array(channel_features)
                t_in = torch.tensor(fft_res,dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
                with torch.no_grad():
                    outputs = model(t_in)
                    softmax_out = nn.Softmax(dim=1)(outputs)
                    _, pred = torch.max(softmax_out,1)
                y_pred_list.append(pred.item()); y_true_list.append(test_labels[i])
                if (i+1)%2==0:
                    predicted_class = int(torch.argmax(softmax_out[0]).item())
                    if softmax_out[0][predicted_class]>pred_thresh:
                        zk_arrs[predicted_class]+=1
                    batch_test = t_in
                    if use_robustbn:
                        if not isinstance(model.bn1,RobustBN):
                            model.bn1 = RobustBN(model.bn1,robustbn_momentum)
                            model.bn2 = RobustBN(model.bn2,robustbn_momentum)
                        model.bn1.train(); model.bn2.train()
                        temp_loader = DataLoader(TensorDataset(batch_test,torch.zeros(batch_test.size(0))),batch_size=batch_test.size(0))
                        update_bn_statistics(model,temp_loader,bn_update_choice,device)
                    model.train()
                    for _ in range(3):
                        outputs_batch = model(batch_test)
                        softmax_out_batch = nn.Softmax(dim=1)(outputs_batch/2.0)
                        CE_loss = torch.mean(Entropy(softmax_out_batch))
                        qk = torch.zeros((n_classes,),dtype=torch.float32,device=device)
                        for k in range(n_classes):
                            qk[k] = softmax_out_batch.mean(dim=0)[k]/(40+zk_arrs[k])
                        normed_qk = qk/torch.sum(qk)
                        R_loss = torch.sum(normed_qk*torch.log(normed_qk+1e-2))
                        loss = CE_loss+R_loss
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                    model.eval()
            correct = sum([1 if y_pred_list[i]==y_true_list[i] else 0 for i in range(len(y_true_list))])
            test_acc = correct/len(y_true_list)
            if test_acc>best_accuracy:
                best_accuracy = test_acc
                best_model = copy.deepcopy(model)
        torch.save(best_model.state_dict(), os.path.join(model_dir, f"CNN_{subject}_best.pth"))
        model = best_model.eval()
        _, _, raw_test_datas, raw_test_labels, _ = data_batch_FFT_one_tester("abs",subject,1,apply_EA=False,return_raw_test=True)
        N = len(raw_test_datas)
        N_adapt = N//2
        adapt_datas = raw_test_datas[:N_adapt]
        holdout_datas = raw_test_datas[N_adapt:]
        holdout_labels = raw_test_labels[N_adapt:]
        R = train_ref.copy()
        zk_arrs = torch.zeros(n_classes,dtype=torch.float32,device=device)
        pred_thresh = 0.7
        optimizer = optim.SGD(model.parameters(),lr=0.00003,momentum=0.9,weight_decay=0.0001)
        acc_list = []
        buffer = []
        for i in tqdm(range(N_adapt), desc=f"{subject} TTA Steps"):
            sample = adapt_datas[i]
            R = EA_online(sample,R,i,5000)
            sqrtRefEA = fractional_matrix_power(R,-0.5)
            aligned_sample = np.dot(sqrtRefEA,sample)
            channel_features = []
            for j in range(aligned_sample.shape[0]):
                sig = aligned_sample[j,:]
                fft_j = np.fft.rfft(sig,n=1250)/window_length
                feat_j = np.abs(fft_j)[slice_start:slice_end]
                channel_features.append(feat_j)
            fft_res = np.array(channel_features)
            buffer.append(fft_res)
            t_adapt = torch.tensor(fft_res,dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = model(t_adapt)
                softmax_out = nn.Softmax(dim=1)(outputs)
            predicted_class = int(torch.argmax(softmax_out[0]).item())
            if softmax_out[0][predicted_class]>pred_thresh:
                zk_arrs[predicted_class]+=1
            if (i+1)%2==0:
                batch_test = t_adapt
                if use_robustbn:
                    if not isinstance(model.bn1,RobustBN):
                        model.bn1 = RobustBN(model.bn1,robustbn_momentum)
                        model.bn2 = RobustBN(model.bn2,robustbn_momentum)
                    model.bn1.train(); model.bn2.train()
                    temp_loader = DataLoader(TensorDataset(batch_test,torch.zeros(batch_test.size(0))),batch_size=batch_test.size(0))
                    update_bn_statistics(model,temp_loader,bn_update_choice,device)
                model.train()
                for _ in range(3):
                    outputs_batch = model(batch_test)
                    softmax_out_batch = nn.Softmax(dim=1)(outputs_batch/2.0)
                    CE_loss = torch.mean(Entropy(softmax_out_batch))
                    qk = torch.zeros((n_classes,),dtype=torch.float32,device=device)
                    for k in range(n_classes):
                        qk[k] = softmax_out_batch.mean(dim=0)[k]/(40+zk_arrs[k])
                    normed_qk = qk/torch.sum(qk)
                    R_loss = torch.sum(normed_qk*torch.log(normed_qk+1e-2))
                    loss = CE_loss+R_loss
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                model.eval()
            correct = 0
            for hs,hl in zip(holdout_datas,holdout_labels):
                sqrtRefEA_h = fractional_matrix_power(R,-0.5)
                aligned_h = np.dot(sqrtRefEA_h,hs)
                channel_features_h = []
                for j in range(aligned_h.shape[0]):
                    sig = aligned_h[j,:]
                    fft_j = np.fft.rfft(sig,n=1250)/window_length
                    feat_j = np.abs(fft_j)[slice_start:slice_end]
                    channel_features_h.append(feat_j)
                fft_res_h = np.array(channel_features_h)
                t_h = torch.tensor(fft_res_h,dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
                with torch.no_grad():
                    out_h = model(t_h)
                    _, p_h = torch.max(nn.Softmax(dim=1)(out_h),1)
                if p_h.item()==hl:
                    correct+=1
            acc_list.append(correct/len(holdout_labels))
        curves[subject] = acc_list
        plt.figure()
        plt.plot(range(1,len(acc_list)+1),acc_list)
        plt.xlabel('Adaptation Step')
        plt.ylabel('Holdout Accuracy')
        plt.title(f'{subject} TTA Curve')
        plt.savefig(f'{subject}_acc_curve.png')
    all_curve = np.mean([curves[s] for s in subjects_to_plot],axis=0)
    plt.figure()
    plt.plot(range(1,len(all_curve)+1),all_curve)
    plt.xlabel('Adaptation Step')
    plt.ylabel('Mean Holdout Accuracy')
    plt.title('S_all TTA Curve')
    plt.savefig('S_all_acc_curve.png')

if __name__=='__main__':
    main()


S2 TTA Steps:  16%|█▋        | 98/600 [04:57<25:15,  3.02s/it]